In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from mpl_toolkits.mplot3d import Axes3D 
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"
import netket as nk

# Import Json, this will be needed to load log files
import json

# Helper libraries
import numpy as np
import matplotlib.pyplot as plt
import time

from flax import nnx
import jax.numpy as jnp
import jax

import json
import pandas as pd
import math 
import sys
import seaborn as sns
import plotly.express as px
from scipy.stats import norm
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, FixedLocator

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np


In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from mpl_toolkits.mplot3d import Axes3D 
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"
import netket as nk
import json
import numpy as np
import matplotlib.pyplot as plt
import time
from flax import nnx
import jax.numpy as jnp
import jax
import json
import pandas as pd
import math 
import sys
import seaborn as sns
import plotly.express as px
from scipy.stats import norm

In [ ]:
def downsample(iters, values, stride=5):
    return iters[::stride], values[::stride]

In [ ]:
colors = ['red', 'green', 'blue', 'black', 'magenta', 'goldenrod']
markers = ['x', 's', 'D', 'v', '*', 'h']
linestyles = ['-', '--', '-.', ':', '-.', '--']

In [ ]:
class Jastrow(nnx.Module):
    def __init__(self, N: int, *, rngs: nnx.Rngs):
        k1, k2 = jax.random.split(rngs.params())
        self.J = nnx.Param(0.01 * jax.random.normal(k1, (N, N),
                                                    dtype=jnp.complex128))

        self.v_bias = nnx.Param(0.01 * jax.random.normal(k2, (N, 1),
                                                         dtype=jnp.complex128))

    def __call__(self, x):
        x = x.astype(jnp.complex128)              # keep the dtypes aligned
        quad = jnp.einsum('...i,ij,...j->...', x, self.J, x)
        lin  = jnp.squeeze(x @ self.v_bias, -1)   # (...,N) @ (N,1) → (...,1)
        return quad + lin

In [ ]:
trained_params_list = []; parameters_list = [];iii = [] 
def conf(J,L):
    # Define custom graph
    edge_colors = []
    for i in range(L):
        edge_colors.append([i, (i + 1) % L, 1])
        edge_colors.append([i, (i + 2) % L, 2])

    # Define the netket graph object
    g = nk.graph.Graph(edges=edge_colors)

    sigmaz = [[1, 0], [0, -1]]
    mszsz = np.kron(sigmaz, sigmaz)

    # Exchange interactions
    exchange = np.asarray([[0, 0, 0, 0], [0, 0, 2, 0], [0, 2, 0, 0], [0, 0, 0, 0]])

    bond_operator = [
        (J[0] * mszsz).tolist(),
        (J[1] * mszsz).tolist(),
        (-J[0] * exchange).tolist(),
        (J[1] * exchange).tolist(),
    ]

    bond_color = [1, 2, 1, 2]
    hi = nk.hilbert.Spin(s=0.5, total_sz=0.0, N=g.n_nodes)
    op = nk.operator.GraphOperator(
        hi, graph=g, bond_ops=bond_operator, bond_ops_colors=bond_color
    )

    return g,hi,op

In [ ]:
def jastrow_calc(T,L,it,v_pbc):
    g  = nk.graph.Hypercube(length=L, n_dim=1, pbc=v_pbc)
    if T == 0:
        hi = nk.hilbert.Spin(s=0.5, N=g.n_nodes)
        ha = (-1)*nk.operator.Heisenberg(hilbert=hi, graph=g)
        mens  = 'The Jastrow ground-state energy is ferromag E0='
    else:
        hi = nk.hilbert.Spin(s=0.5, total_sz=0, N=g.n_nodes)
        ha = nk.operator.Heisenberg(hilbert=hi, graph=g)
        mens  = 'The Jastrow ground-state energy is antiferromag E0='

    ma = Jastrow(N=hi.size, rngs=nnx.Rngs(0))

    if T == 0:
        sa = nk.sampler.MetropolisLocal(hilbert=hi)
        sr = nk.optimizer.SR(diag_shift=0.1, holomorphic=False)
    else:
        sa = nk.sampler.MetropolisExchange(hilbert=hi,graph=g)
        sr = nk.optimizer.SR(diag_shift=0.1, holomorphic=True)   
    
    op = nk.optimizer.Sgd(learning_rate=0.01)
    
    vs = nk.vqs.MCState(sa, ma, n_samples=1008)
    gs = nk.VMC(
        hamiltonian=ha,
        optimizer=op,
        preconditioner=sr,
        variational_state=vs)
    start = time.time()
    j_out = 'dataf/jastrow_' + str(L) + '_' + str(T) + '_' + str(it)
    gs.run(it, out=j_out)
    end   = time.time()
    final_energy = float(gs.energy.mean.real)
    return final_energy,j_out

In [ ]:
def exac_calc_lanczos_ed(T,L,vpbc):
    g = nk.graph.Hypercube(length=L, n_dim=1, pbc=vpbc)
    if T == 0 :
        hi = nk.hilbert.Spin(s=0.5, N=g.n_nodes)
        ha = -1.0 * nk.operator.Heisenberg(hilbert=hi, graph=g)
        mens = 'The exact ground-state energy is ferromag E0='
    else:
        hi = nk.hilbert.Spin(s=0.5, total_sz=0, N=g.n_nodes)
        ha = nk.operator.Heisenberg(hilbert=hi, graph=g)
        mens = 'The exact ground-state energy is anti-ferromag E0='
        
    evals = nk.exact.lanczos_ed(ha, compute_eigenvectors=False)
    exact_gs_energy = evals[0]

    exact_df = pd.DataFrame()
    exact_v  = []
    exact_v.append(float(exact_gs_energy))
    exact_row_df = pd.DataFrame([exact_v])
    exact_df = pd.concat([exact_row_df])
    exact_df.insert(0, 'id', range(1, 1 + len(exact_df)))
    exact_df.columns = ['id','value']
    e_path = "dataf/exact_" + str(L)  + "_"  + str(T) + ".csv"

    exact_df.to_csv(e_path)

    return e_path, mens, exact_gs_energy 

In [ ]:
def info(e):
    head   = list(e.keys())[0]
    body   = list(e[head].keys())
    bias   = e[head][body[0]]
    kernel = e[head][body[1]]
    return  head, body, list(bias), list(kernel)
def real(c):
    return float(np.real(c))  
def img(c):
    return float(np.imag(c))    
def r_i(c):
    return real(c),img(c)  

def save_params(step, params, energy):
    trained_params_list.append(params.copy())
    parameters_list.append(energy.state.parameters.copy())
    iii.append(1)
    return True

In [ ]:
def srbm(T,L,it,vp,va):

    paths = []

    g  = nk.graph.Hypercube(length=L, n_dim=1, pbc=vp)
    if T == 0:
        hi = nk.hilbert.Spin(s=0.5, N=g.n_nodes)
        ha = (-1)*nk.operator.Heisenberg(hilbert=hi, graph=g)
        mens  = 'The RBM ground-state energy is ferromag E0='
        sa = nk.sampler.MetropolisLocal(hilbert=hi)
        print(va)
        ma = nk.models.RBM(alpha=va) 
        
        
    else:
        hi = nk.hilbert.Spin(s=0.5, total_sz=0, N=g.n_nodes)
        ha = nk.operator.Heisenberg(hilbert=hi, graph=g)
        mens  = 'The RBM ground-state energy is antiferromag E0='

        sa = nk.sampler.MetropolisExchange(hilbert=hi,graph=g)
        ma = nk.models.RBMSymm(symmetries=g.translation_group(), alpha=va)
      
    
    
    op = nk.optimizer.Sgd(learning_rate=0.01)
    sr = nk.optimizer.SR(diag_shift=0.1, holomorphic=False)
    vs = nk.vqs.MCState(sa, ma, n_samples=1008)

    opt = nk.optimizer.Sgd(learning_rate=0.01)

    sr = nk.optimizer.SR(diag_shift=0.01)
       
    gs = nk.VMC(
        hamiltonian=ha,
        optimizer=op,
        preconditioner=sr,
        variational_state=vs)
    start = time.time()
    r_out = 'dataf/srbm_' + str(L) + '_' + str(T) + '_' + str(it) 
    gs.run(out=r_out, 
           n_iter=it,
           save_params_every=1,
          callback=save_params)
    ### <b> Results </b>
    head, body, bias_list,kernel_list = info(parameters_list[-1])

    real_df = pd.DataFrame()
    img_df  = pd.DataFrame()
    for param in parameters_list:
        head, body, bias_list,kernel_list = info(param)
        real_v = [];img_v = []
        for bias in bias_list:
            nr, ni = r_i(bias); 
            real_v.append(nr)
            img_v.append(ni)   
    
        real_row_df = pd.DataFrame([real_v])
        img_row_df  = pd.DataFrame([img_v])
        
        real_df = pd.concat([real_df,real_row_df])
        img_df  = pd.concat([img_df,img_row_df])   


    real_df.insert(0, 'id', range(1, 1 + len(real_df)))
    img_df.insert(0, 'id', range(1, 1 + len(img_df)))

    path_nm = "dataf/srbm_" + str(L) +  '_' + str(T)  + "_"  + str(it)
    real_df.to_csv(path_nm + "_bias_real.csv",index=None) 
    paths.append(path_nm + "_bias_real.csv")
    img_df.to_csv(path_nm + "_bias_img.csv",index=None) 
    paths.append(path_nm + "_bias_img.csv")



    real_kernel_df = pd.DataFrame()
    img_kernel_df  = pd.DataFrame()
    for param in parameters_list:
        head, body, bias_list,kernel_list = info(param)
        real_v = [];img_v = []
        for ks in kernel_list:
            for k in ks.flatten():
                nr, ni = r_i(k); 
                real_v.append(nr)
                img_v.append(ni) 
        real_row_df = pd.DataFrame([real_v])
        img_row_df  = pd.DataFrame([img_v])
    
        real_kernel_df = pd.concat([real_kernel_df,real_row_df])
        img_kernel_df  = pd.concat([img_kernel_df,img_row_df])  

    real_kernel_df.insert(0, 'id', range(1, 1 + len(real_kernel_df)))
    img_kernel_df.insert(0, 'id', range(1, 1 + len(img_kernel_df)))

    path_nm = "dataf/srbm_" + str(L) +  '_' + str(T)  + "_"  + str(it)
    real_kernel_df.to_csv(path_nm + "_kernel_real.csv",index=None) 
    paths.append(path_nm + "_kernel_real.csv")
    img_kernel_df.to_csv(path_nm + "_kernel_img.csv",index=None) 
    paths.append(path_nm + "_kernel_img.csv")


    end = time.time()
    return gs,r_out,paths 

In [ ]:
def plot(e_path1, e_path2, j_out1, r_out1, j_out2, r_out2):
    fig, axs = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

    j_out1 = j_out1 + ".log"
    r_out1 = r_out1 + ".log"
    

    # ========== Subplot (a) ==========
    ax = axs[0]


    df = pd.read_csv(e_path1)
    exact_gs_energy = df.iloc[0, 2]


    fx = j_out1
    with open(fx) as f:
        data = json.load(f)

    iters_Jastrow = data["Energy"]["iters"]
    energy_raw    = data["Energy"]["Mean"]
    if isinstance(energy_raw, list):
        energy_Jastrow = [e["real"] for e in energy_raw]
    else:
        energy_Jastrow = energy_raw["real"]

    stride = 5
    iters_Jastrow_ds, energy_Jastrow_ds = downsample(iters_Jastrow, energy_Jastrow, stride)  
   
    fx = r_out1
    with open(fx) as f:
        data = json.load(f)
    
    iters_RBM   = data["Energy"]["iters"]
    energy_RBM  = data["Energy"]["Mean"]                
    iters_RBM_ds, energy_RBM_ds = downsample(iters_RBM, energy_RBM, stride)
    ltp = "RBM"

    stride = 5
    iters_RBM, energy_RBM_ds = downsample(iters_RBM, energy_RBM, stride)        

    label = r"RBM"
    i = 0
    ax.plot(iters_RBM_ds, energy_RBM_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)
    i = 1
    label = r"JASTROW"
    ax.plot(iters_Jastrow_ds, energy_Jastrow_ds, label=label, linestyle=linestyles[i],
                marker=markers[i], color=colors[i], markersize=5)


    if (exact_gs_energy != 0):
        ax.axhline(y=exact_gs_energy, color='k', lw=2, ls='--', label='Exact')

    # Configurações do eixo
    ax.tick_params(direction='in', length=4, width=1.0, top=True, bottom=True, left=True, right=True)
    ax.tick_params(which='minor', direction='in', length=2, width=0.8, top=True, bottom=True, left=True, right=True)
    ax.tick_params(labelbottom=False)
    from matplotlib.ticker import MaxNLocator
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.yaxis.set_minor_locator(FixedLocator([]))
    ax.set_ylabel(r"Energia")
    ax.text(0.03, 0.94, '(a) $L=22$' + ' Regime Ferromagnético', transform=ax.transAxes, fontsize=12, verticalalignment='top')
    ax.legend(fontsize=9, loc='upper right', frameon=False)

    # ========== Subplot (c) ==========
    ax = axs[1]


    j_out2  = j_out2 + ".log"
    r_out2  = r_out2 + ".log"
    df = pd.read_csv(e_path2)
    exact_gs_energy = df.iloc[0, 2]

    fx = j_out2
    with open(fx) as f:
        data = json.load(f)

    iters_Jastrow = data["Energy"]["iters"]
    energy_raw    = data["Energy"]["Mean"]
    if isinstance(energy_raw, list):
        energy_Jastrow = [e["real"] for e in energy_raw]
    else:
        energy_Jastrow = energy_raw["real"]

    stride = 5
    iters_Jastrow_ds, energy_Jastrow_ds = downsample(iters_Jastrow, energy_Jastrow, stride)  
    fx = r_out2
    with open(fx) as f:
        data = json.load(f)
    
    iters_RBM   = data["Energy"]["iters"]
    energy_RBM  = data["Energy"]["Mean"]                
    iters_RBM_ds, energy_RBM_ds = downsample(iters_RBM, energy_RBM, stride)
    ltp = "SRBM"

    stride = 5
    iters_RBM, energy_RBM_ds = downsample(iters_RBM, energy_RBM, stride) 


    label = r"SRBM"
    i = 0
    ax.plot(iters_RBM_ds, energy_RBM_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)
    i = 1
    label = r"JASTROW"
    ax.plot(iters_Jastrow_ds, energy_Jastrow_ds, label=label, linestyle=linestyles[i],
                marker=markers[i], color=colors[i], markersize=5)


    if (exact_gs_energy!=0):
            ax.axhline(y=exact_gs_energy, color='k', lw=2, ls='--', label='Exact')

    # Configurações do eixo
    ax.tick_params(direction='in', length=4, width=1.0, top=True, bottom=True, left=True, right=True)
    ax.tick_params(which='minor', direction='in', length=2, width=0.8, top=True, bottom=True, left=True, right=True)
    from matplotlib.ticker import MaxNLocator
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    ax.yaxis.set_major_locator(MaxNLocator(integer=True))

    ax.set_ylabel(r"Energia")
    ax.set_xlabel(r"Interações")
    ax.text(0.03, 0.94, '(b) $L=22' + ' Regime Antiferromagnético', transform=ax.transAxes, fontsize=12, verticalalignment='top')
    ax.legend(fontsize=9, loc='upper right', frameon=False)


    # Layout final
    plt.subplots_adjust(hspace=0.1, wspace=0.1)
    plt.savefig("L_22_0_1.png", dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

In [ ]:
def plotdx(paths, ip, lp):
    # ==============================================
    # BLOCK 1: Data Loading and Initial Processing
    # ==============================================

    print(paths)
    print(paths[2])
    
    df = pd.read_csv(paths[2])
    i_line = df.iloc[ip].iloc[1:]  # First row, excluding first column
    l_line = df.iloc[lp].iloc[1:]  # Last row, excluding first column

    # ==============================================
    # BLOCK 2: Difference Calculations
    # ==============================================
    dif = l_line - i_line  # Calcula a diferença vetorial
    
    mod_dif = np.linalg.norm(dif)  # Sempre calcula a norma
    
    # ==============================================
    # BLOCK 3: Main Visualization (2x2 Grid)
    # ==============================================
    plt.figure(figsize=(18, 6))
    
    # Subplot 1: Initial Distribution
    ax1 = plt.subplot(1, 2, 1)
    plot_with_normal(ax1, i_line, 'orange', 'Distribuição Inicial com Ajuste Normal')
    ax1.set_xticks([])  # Remove os ticks do eixo x
    ax1.set_xlabel('')  # Remove o label do eixo x
    # Subplot 2: Final Distribution
    ax2 = plt.subplot(1, 2, 2)
    plot_with_normal(ax2, l_line, 'crimson', 'Distribuição Final com Ajuste Normal')
    ax2.set_xticks([])  # Remove os ticks do eixo x
    ax2.set_xlabel('')  # Remove o label do eixo x

       
    plt.tight_layout()
    path_img = "dist_w/s1.png"
    plt.savefig(path_img)

    plt.show()

    # ==============================================
    # BLOCK 4: Secondary Visualization (Bar and Box Plots)
    # ==============================================
    plt.figure(figsize=(18, 6))
    
    # Subplot 1: Bar Plot of Differences
    ax1 = plt.subplot(1, 2, 1)
    bars = ax1.bar(dif.index, dif.values, 
                 color=np.where(dif.values >= 0, 'green', 'red'),
                 alpha=0.6, edgecolor='black')
    
    ax1.axhline(0, color='black', linewidth=0.8)
    ax1.set_title('Diferenças Individuais: Final - Inicial', fontsize=14)
    ax1.set_xlabel('Variáveis', fontsize=12)
    ax1.set_ylabel('Diferença', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    ax1.grid(axis='y', alpha=0.2)

    # Add value labels
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height/2,
                f'{height:.1f}',
                ha='center', va='center',
                fontsize=9, fontweight='bold')

    # Subplot 2: Boxplot Comparison
    ax2 = plt.subplot(1, 2, 2)
    sns.boxplot(data=[i_line, l_line], 
              palette=['orange', 'crimson'],
              width=0.4,
              showmeans=True,
              meanprops={"marker":"o", "markerfacecolor":"white", "markeredgecolor":"black"},
              ax=ax2)

    ax2.set_title('Comparação por Estatísticas Descritivas', fontsize=14)
    ax2.set_xticklabels(['Linha Inicial', 'Linha Final'])
    ax2.set_ylabel('Valores', fontsize=12)
    
    # Add mean annotations
    ax2.text(0, i_line.mean(), f"Média: {i_line.mean():.2f}", 
           ha='center', va='bottom', fontsize=10)
    ax2.text(1, l_line.mean(), f"Média: {l_line.mean():.2f}", 
           ha='center', va='bottom', fontsize=10)

    ax2.grid(axis='y', alpha=0.2)
    plt.tight_layout()
    path_img = "dist_w/s2.png"
    plt.savefig(path_img)
    plt.show()

    # ==============================================
    # BLOCK 4: Additional Comparative Plots
    # ==============================================
    plt.figure(figsize=(12, 4))  # Tamanho consistente com os outros blocos

    # Gráfico 1: Comparação ponto a ponto
    ax1 = plt.subplot(1, 2, 2)
    # Plot dos pontos e linhas
    ax1.scatter(i_line.index, i_line.values, color='orange', alpha=0.7, label='Série Inicial')
    ax1.scatter(l_line.index, l_line.values, color='crimson', alpha=0.7, label='Série Final')
    ax1.plot(i_line.index, i_line.values, 'orange', linestyle=':', alpha=0.4)
    ax1.plot(l_line.index, l_line.values, 'crimson', linestyle=':', alpha=0.4)

    # Linhas de conexão
    for idx in i_line.index:
        ax1.plot([idx, idx], [i_line[idx], l_line[idx]], 'gray', linestyle='--', alpha=0.3)

    ax1.set_title('Comparação Ponto a Ponto', fontsize=12)
    ax1.set_xlabel('Variáveis', fontsize=10)
    ax1.set_ylabel('Valores', fontsize=10)
    ax1.legend(fontsize=9)
    ax1.grid(alpha=0.2)
    plt.xticks(rotation=45, ha='right')  # Melhor alinhamento para rótulos

    # Gráfico 2: Comparação de distribuições
    ax2 = plt.subplot(1, 2, 1)
    # KDE plots
    sns.kdeplot(i_line, color='orange', label='Inicial', fill=True, alpha=0.3, ax=ax2)
    sns.kdeplot(l_line, color='crimson', label='Final', fill=True, alpha=0.3, ax=ax2)
    # Linhas de média
    ax2.axvline(i_line.mean(), color='orange', linestyle='--', 
               label=f'Média Inicial: {i_line.mean():.2f}')
    ax2.axvline(l_line.mean(), color='crimson', linestyle='--', 
               label=f'Média Final: {l_line.mean():.2f}')

    ax2.set_title('Comparação das Distribuições', fontsize=12)
    ax2.set_xlabel('Valores', fontsize=10)
    ax2.set_ylabel('Densidade', fontsize=10)
    ax2.legend(fontsize=9)
    ax2.grid(alpha=0.2)

    plt.tight_layout()
    plt.savefig("dist_w/s0.png", dpi=300, bbox_inches='tight')



    plt.show()

    plt.close()
    # ==============================================
    # BLOCK 5: Statistical Metrics
    # ==============================================
    mean_diff = dif.mean()
    std_diff = dif.std()
    effect_size = mean_diff / std_diff  # Cohen's d

    print(
        f"\nMétricas Estatísticas:\n"
        f"----------------------\n"
        f"Média da diferença: {mean_diff:.2f}\n"
        f"Desvio padrão da diferença: {std_diff:.2f}\n"
        f"Tamanho do efeito (Cohen's d): {effect_size:.2f}\n"
        f"Magnitude da diferença (norma L2): {mod_dif:.2f}")



        
            
        # Extract points
        initial_point = df.iloc[initial_point_idx, 1:]  # Exclude first column
        final_point = df.iloc[last_point_idx, 1:]
        all_points = df.iloc[:, 1:].values
        
        if len(initial_point) != len(final_point):
            raise ValueError("Initial and final series must have the same length")

        from sklearn.impute import SimpleImputer

        imputer = SimpleImputer(strategy="mean")  # ou "median", "most_frequent", etc.
        X_imputed = imputer.fit_transform(X)
        
        # PCA Analysis
        X_std = StandardScaler().fit_transform(all_points)
        n_components = min(2, all_points.shape[1])
        pca = PCA(n_components=n_components)
        principal_components = pca.fit_transform(X_std)
        distance = np.linalg.norm(principal_components[initial_point_idx] - 
                                 principal_components[last_point_idx])

        

In [ ]:
def plotdp(paths, initial_point_idx, last_point_idx):
    save_path="dist_w/s4.png"
    """
    Plots the dynamics between initial and final points using heatmap and PCA trajectory.
    
    Parameters:
    - paths: List of file paths (uses index 2)
    - initial_point_idx: Index of initial point in the dataframe
    - last_point_idx: Index of final point in the dataframe
    - save_path: Path to save the output figure
    """
    try:
        # Data loading and validation
        df = pd.read_csv(paths[2])
        if df.empty:
            raise ValueError("Dataframe is empty")

    
        # Extract points
        initial_point = df.iloc[initial_point_idx, 1:]  # Exclude first column
        final_point = df.iloc[last_point_idx, 1:]
        all_points = df.iloc[:, 1:].values  # Todas as séries (sem a primeira coluna)

        # Verificação de consistência
        if len(initial_point) != len(final_point):
            raise ValueError("Initial and final series must have the same length")

        # Trata valores ausentes (NaN) com média da coluna
        imputer = SimpleImputer(strategy="mean")
        all_points_imputed = imputer.fit_transform(all_points)

        # Padronização (zero média, desvio padrão 1)
        X_std = StandardScaler().fit_transform(all_points_imputed)

        # PCA
        n_components = min(2, X_std.shape[1])
        pca = PCA(n_components=n_components)
        principal_components = pca.fit_transform(X_std)

        # Distância Euclidiana entre o ponto inicial e final no espaço PCA
        distance = np.linalg.norm(
            principal_components[initial_point_idx] - principal_components[last_point_idx]
        )

        # Create figure
        fig, axs = plt.subplots(1, 2, figsize=(16, 6))
        # Plot 1: Heatmap of differences
        diff_df = pd.DataFrame({
            'Initial': initial_point,
            'Final': final_point,
            'Difference': final_point - initial_point
        })
        
        sns.heatmap(diff_df.T, annot=True, fmt=".1f", cmap="coolwarm", center=0,
                   ax=axs[0], cbar_kws={'label': 'Difference Magnitude'})
        axs[0].set_title("Variable Differences Heatmap")
        axs[0].set_xlabel("Variables")
        axs[0].set_ylabel("Comparison")

        # Plot 2: PCA Trajectory
        if n_components >= 2:
            # Full trajectory
            axs[1].scatter(principal_components[:, 0], principal_components[:, 1], 
                          alpha=0.3, label='Intermediate Points')
            axs[1].plot(principal_components[:, 0], principal_components[:, 1], 
                       'grey', alpha=0.3, label='Trajectory')
            
            # Highlight points
            axs[1].scatter(principal_components[initial_point_idx, 0], 
                          principal_components[initial_point_idx, 1], 
                          color='green', s=100, label='Start Point')
            axs[1].scatter(principal_components[last_point_idx, 0], 
                          principal_components[last_point_idx, 1], 
                          color='red', s=100, label='End Point')
            
            axs[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
            axs[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
        else:
            # 1D case
            axs[1].scatter(principal_components[:, 0], np.zeros(len(principal_components)), 
                          alpha=0.3, label='Intermediate Points')
            axs[1].plot(principal_components[:, 0], np.zeros(len(principal_components)), 
                       'grey', alpha=0.3, label='Trajectory')
            axs[1].scatter(principal_components[initial_point_idx, 0], 0, 
                          color='green', s=100, label='Start Point')
            axs[1].scatter(principal_components[last_point_idx, 0], 0, 
                          color='red', s=100, label='End Point')
            axs[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')

        axs[1].set_title(f'PCA Trajectory (Distance: {distance:.2f})')
        axs[1].legend()
        axs[1].grid(alpha=0.3)

        plt.tight_layout()
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        plt.show()
        plt.close()
        print(f"Figure saved to {save_path}")
        
    except Exception as e:
        print(f"Error in plot_dynamics: {str(e)}")
        raise

In [ ]:
# Função para plotar com overlay de curva normal
def plot_with_normal(ax, data, color, title):
    sns.histplot(data, kde=True, color=color, ax=ax, stat='density', alpha=0.4)
    mu, std = norm.fit(data)
    xmin, xmax = ax.get_xlim()
    x = np.linspace(xmin, xmax, 100)
    p = norm.pdf(x, mu, std)
    ax.plot(x, p, color=color, linewidth=2)
    ax.set_title(title)
    ax.grid(alpha=0.3)